# Windows Desktop Automation Agent (sandbox)

**Stack:** LM Studio / Ollama · `windows-mcp` · Python MCP SDK · ReAct loop

### Quick-start
1. Install `uv` on Windows: `winget install astral-sh.uv` or `pip install uv`
2. Set `LMSTUDIO_BASE_URL` or `OLLAMA_BASE_URL` in `.env`
3. Edit `SYSTEM_PROMPT` (Cell 5) and `TASK` (Cell 8)
4. Run all cells

> **Requirements:** Python 3.13+, `uv` in PATH


In [77]:
%pip install -q mcp httpx python-dotenv nest_asyncio Pillow APScheduler ipywidgets pyyaml


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [78]:
# ── Imports ───────────────────────────────────────────────────────────────────
import asyncio, base64, io, json, os, shutil, textwrap
from datetime import datetime
from pathlib import Path

import httpx
import ipywidgets as widgets
import nest_asyncio
from dotenv import load_dotenv
from IPython.display import display, Image as IPImage, Markdown
from PIL import Image as PILImage

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()
load_dotenv()

print("✓ Imports OK")

✓ Imports OK


In [79]:
# ── Config (inline — no external config.json needed) ────────────────────────
_cfg = {
    "backend":                "lmstudio",  # "ollama" | "lmstudio"
    "model":                  "qwen3-coder:30b",
    "max_steps":              20,
    "use_vision":             "auto",   # "false" | "true" | "auto"
    "flash_mode":             True,

    "save_artifacts":         True,
    "save_gif":               True,
    "gif_frame_ms":           800,

    "human_in_loop":          False,
    "confirm_keywords":       ["delete", "format", "shutdown", "restart", "rm", "remove"],

    "retry_enabled":          True,
    "max_retries":            3,
    "retry_delay_s":          2,

    "context_compress_after": 30,

    "loop_detection":         True,
    "loop_window":            5,
    "max_loop_repeats":       3,

    "model_timeout_s":        120,
    "max_subagents":          4,
    "schedule":               None,
}

# ── Core ──────────────────────────────────────────────────────────────────────
MODEL      = _cfg["model"]
MAX_STEPS  = int(_cfg["max_steps"])
USE_VISION = str(_cfg["use_vision"])
FLASH_MODE = bool(_cfg["flash_mode"])

assert USE_VISION in {"false", "true", "auto"}, (
    f"use_vision must be 'false', 'true', or 'auto' — got {USE_VISION!r}"
)

OLLAMA_URL   = os.getenv("OLLAMA_BASE_URL",   "http://localhost:11434").rstrip("/")
LMSTUDIO_URL = os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234").rstrip("/")
BACKEND      = _cfg["backend"].lower()

# ── Artifacts & GIF ───────────────────────────────────────────────────────────
SAVE_ARTIFACTS = bool(_cfg["save_artifacts"])
SAVE_GIF       = bool(_cfg["save_gif"])
GIF_FRAME_MS   = int(_cfg["gif_frame_ms"])

# ── Human-in-the-loop ─────────────────────────────────────────────────────────
HUMAN_IN_LOOP    = bool(_cfg["human_in_loop"])
CONFIRM_KEYWORDS = list(_cfg["confirm_keywords"])

# ── Retry & error recovery ────────────────────────────────────────────────────
RETRY_ENABLED = bool(_cfg["retry_enabled"])
MAX_RETRIES   = int(_cfg["max_retries"])
RETRY_DELAY_S = float(_cfg["retry_delay_s"])

# ── Context compression ───────────────────────────────────────────────────────
CONTEXT_COMPRESS_AFTER = int(_cfg["context_compress_after"])

# ── Loop detection ────────────────────────────────────────────────────────────
LOOP_DETECTION   = bool(_cfg["loop_detection"])
LOOP_WINDOW      = int(_cfg["loop_window"])
MAX_LOOP_REPEATS = int(_cfg["max_loop_repeats"])

# ── Model timeout ─────────────────────────────────────────────────────────────
MODEL_TIMEOUT_S = int(_cfg["model_timeout_s"])

# ── Orchestrator / Scheduler ──────────────────────────────────────────────────
MAX_SUBAGENTS = int(_cfg["max_subagents"])
SCHEDULE      = _cfg["schedule"]

# ── Windows-MCP server ────────────────────────────────────────────────────────
_UVX_SEARCH_PATHS = [
    os.path.expanduser(r"~\.local\bin\uvx.exe"),
    os.path.expanduser(r"~\.cargo\bin\uvx.exe"),
    r"C:\Users\Public\.local\bin\uvx.exe",
]
_uvx = shutil.which('uvx') or shutil.which('uvx.exe')
if _uvx is None:
    for _p in _UVX_SEARCH_PATHS:
        if os.path.isfile(_p) and os.access(_p, os.X_OK):
            _uvx = _p
            break
if _uvx is None:
    raise EnvironmentError(
        "uvx not found.\n"
        "  Windows: winget install astral-sh.uv\n"
        "  or: pip install uv\n"
        "  or visit https://docs.astral.sh/uv/"
    )

SERVER_PARAMS = StdioServerParameters(command=_uvx, args=['windows-mcp'], env=dict(os.environ))

print(f'  model           : {MODEL}')
print(f'  max_steps       : {MAX_STEPS}  |  use_vision: {USE_VISION}')
print(f'  flash_mode      : {FLASH_MODE}  →  think={not FLASH_MODE}')
print(f'  save_artifacts  : {SAVE_ARTIFACTS}  |  save_gif: {SAVE_GIF}  |  gif_frame_ms: {GIF_FRAME_MS}')
print(f'  human_in_loop   : {HUMAN_IN_LOOP}  |  keywords: {CONFIRM_KEYWORDS}')
print(f'  retry_enabled   : {RETRY_ENABLED}  |  max_retries: {MAX_RETRIES}  |  delay: {RETRY_DELAY_S}s')
print(f'  context_compress: after {CONTEXT_COMPRESS_AFTER} turns  (0=off)')
print(f'  loop_detection  : {LOOP_DETECTION}  |  window: {LOOP_WINDOW}  |  max_repeats: {MAX_LOOP_REPEATS}')
print(f'  model_timeout_s : {MODEL_TIMEOUT_S}  (0=disabled)')
print(f'  max_subagents   : {MAX_SUBAGENTS}')
print(f'  schedule        : {SCHEDULE or "off (run once)"}')
print(f'  backend         : {BACKEND}  |  url: {OLLAMA_URL if BACKEND == "ollama" else LMSTUDIO_URL}')
print(f'  uvx             : {_uvx}')


  model           : qwen3-coder:30b
  max_steps       : 20  |  headless: False  |  use_vision: false
  flash_mode      : True  →  think=False
  session_file    : None
  save_artifacts  : True  |  save_gif: True  |  gif_frame_ms: 800
  human_in_loop   : False  |  keywords: ['submit', 'buy', 'delete', 'send', 'login']
  retry_enabled   : True  |  max_retries: 3  |  delay: 2.0s
  context_compress: after 30 turns  (0=off)
  loop_detection  : True  |  window: 5  |  max_repeats: 3
  model_timeout_s : 120  (0=disabled)
  max_subagents   : 6
  schedule        : off (run once)
  backend         : lmstudio  |  url: http://localhost:1234
  npx             : /opt/homebrew/bin/npx


## System Prompt

Edit the cell below to define the agent's persona, constraints, and strategy.

In [80]:
SYSTEM_PROMPT = """\
You are Jerry, an autonomous Windows desktop automation agent.
You control a Windows PC using mouse, keyboard, screenshot, and system tools.

Guidelines:
- Think step-by-step before each action.
- After each observation, assess your progress and decide the next action.
- Use screenshots to verify the current state of the screen before and after actions.
- Prefer the most direct path to completing the task; avoid unnecessary clicks.
- When you have fully completed the task, respond with a clear summary — do NOT call any more tools.
- If you encounter an unrecoverable error, explain it clearly and stop.
"""


In [81]:
# ── Orchestrator system prompt ────────────────────────────────────────────────
# Used by run_orchestrator() when ORCHESTRATOR_TASK is set.
# Edit this to tune how the orchestrator decomposes and delegates goals.

ORCHESTRATOR_SYSTEM_PROMPT = """
You are an orchestration agent. You do not browse the web yourself.
Your job is to break a high-level goal into concrete browser sub-tasks,
delegate them to isolated sub-agents via the spawn_subagents tool, and
synthesise their results into a clear final answer.

Guidelines:
- Use mode="parallel" when sub-tasks are independent (e.g. checking multiple sites simultaneously).
- Use mode="sequential" when a later task depends on an earlier result (each agent receives prior context).
- Keep each sub-task self-contained and specific enough for an agent to complete autonomously.
- After receiving sub-agent results, synthesise them into a concise final answer.
- Do not spawn sub-agents unnecessarily — if the goal is simple, answer directly without calling spawn_subagents.
- You may call spawn_subagents multiple times across your steps, mixing parallel and sequential batches.
""".strip()


## Task

Set what you want the agent to do, then run the **Run Agent** cell at the bottom.

In [82]:
# ── Task ─────────────────────────────────────────────────────────────────────
# Edit TASK to describe what you want Jerry to do on the Windows desktop.

TASK = "Open Notepad, type 'Hello from Jerry!', then save the file to the Desktop as hello.txt"

# Orchestrator mode: set a high-level goal here (leave None for single-agent mode)
ORCHESTRATOR_TASK = None
SUBAGENT_MODE = "sequential"

# Skill mode: not configured for this sandbox (use TASK above)
SKILL      = None
SKILL_ARGS = {}


In [83]:
# ── Skills & Credentials ──────────────────────────────────────────────────────
# Skills are reusable, parameterised task templates defined in skills/<name>.yaml.
# Set SKILL + SKILL_ARGS in the cell above to use one, or leave SKILL = None to
# use the raw TASK string directly (original behaviour — fully backward-compatible).
#
# Credentials are stored in credentials/<name>.yaml (gitignored).
# Reference individual values anywhere in TASK or SKILL_ARGS with:
#   creds:<name>.<key>   e.g.  creds:github.username

import re
import yaml

SKILL      = None   # e.g. "login"  ← override in cell above
SKILL_ARGS = {}     # e.g. {"url": "...", "username": "creds:github.username", ...}

_SKILLS_DIR = Path("skills")
_CREDS_DIR  = Path("credentials")
_CREDS_RE   = re.compile(r'creds:([A-Za-z0-9_-]+)\.([A-Za-z0-9_-]+)')


def _expand_creds(text: str) -> str:
    """
    Replace all creds:name.key tokens in text with values from credentials/<name>.yaml.

    Raises FileNotFoundError if the credential file is missing,
    KeyError if the key is not present in the file.
    """
    _cache: dict[str, dict] = {}

    def _sub(m: re.Match) -> str:
        name, key = m.group(1), m.group(2)
        if name not in _cache:
            path = _CREDS_DIR / f"{name}.yaml"
            if not path.exists():
                raise FileNotFoundError(
                    f"Credential set '{name}' not found at {path}.\n"
                    f"Create credentials/{name}.yaml with the keys your task needs.\n"
                    f"See credentials.example/ for the expected format."
                )
            with path.open() as fh:
                _cache[name] = yaml.safe_load(fh) or {}
        if key not in _cache[name]:
            raise KeyError(
                f"Key '{key}' not found in credentials/{name}.yaml\n"
                f"Available keys: {list(_cache[name].keys())}"
            )
        return str(_cache[name][key])

    return _CREDS_RE.sub(_sub, text)


def load_skill(name: str, args: dict) -> tuple[str, str | None]:
    """
    Load skills/<name>.yaml, validate required args, and return
    (resolved_task, effective_system_prompt_or_None).

    resolved_task            — task_template with all {variables} filled in
    effective_system_prompt  — SYSTEM_PROMPT + system_prompt_extension (if any),
                               or None if no extension (run_agent falls back to global)

    creds:name.key references in args values are resolved before template substitution.

    Raises:
      FileNotFoundError — skill file not found, or referenced credential file not found
      KeyError          — referenced credential key not in file
      ValueError        — required {variable} missing from args
    """
    skill_path = _SKILLS_DIR / f"{name}.yaml"
    if not skill_path.exists():
        available = [p.stem for p in _SKILLS_DIR.glob("*.yaml")] if _SKILLS_DIR.exists() else []
        raise FileNotFoundError(
            f"Skill '{name}' not found at {skill_path}.\n"
            f"Available skills: {available or '(none — is skills/ present?)'}"
        )

    with skill_path.open() as fh:
        skill = yaml.safe_load(fh)

    template: str = skill.get("task_template", "")
    if not template:
        raise ValueError(f"Skill '{name}' has no task_template defined in {skill_path}")

    # Resolve creds:name.key references in arg values before template substitution
    resolved_args = {k: _expand_creds(v) if isinstance(v, str) else v for k, v in args.items()}

    missing = set(re.findall(r'\{(\w+)\}', template)) - set(resolved_args.keys())
    if missing:
        arg_docs = skill.get("args", {})
        raise ValueError(
            f"Skill '{name}' is missing required arguments: {sorted(missing)}\n"
            + "\n".join(f"  • {k}: {arg_docs.get(k, '(no description)')}" for k in sorted(missing))
        )

    resolved_task = template.format(**resolved_args)
    extension     = (skill.get("system_prompt_extension") or "").strip()
    effective_prompt = (SYSTEM_PROMPT.rstrip() + "\n\n" + extension) if extension else None

    print(f"[skill] Loaded '{name}'" + (f" — {skill['description']}" if skill.get("description") else ""))
    return resolved_task, effective_prompt


def _resolve_task() -> tuple[str, str | None]:
    """Return (resolved_task, effective_prompt_or_None) from SKILL/SKILL_ARGS or raw TASK."""
    if SKILL is not None:
        return load_skill(SKILL, SKILL_ARGS)
    return _expand_creds(TASK), None


print("✓ Skills & credentials loader OK")


✓ Skills & credentials loader OK


In [84]:
# ── LLM API ───────────────────────────────────────────────────────────────────

async def llm_chat(
    messages: list[dict],
    tools: list[dict] | None = None,
    format: dict | None = None,
) -> dict:
    """Send a chat request to the configured backend (Ollama or LM Studio).

    Always returns a dict with a top-level "message" key so all callers work
    identically regardless of backend.
    """
    _timeout = MODEL_TIMEOUT_S if MODEL_TIMEOUT_S > 0 else None
    async with httpx.AsyncClient(timeout=_timeout) as client:

        if BACKEND == "lmstudio":
            body: dict = {"model": MODEL, "messages": messages, "stream": False}
            if tools:
                body["tools"] = tools
            resp = await client.post(f"{LMSTUDIO_URL}/v1/chat/completions", json=body)
            resp.raise_for_status()
            data = resp.json()
            # Normalise to Ollama-style {"message": {...}} so callers are unchanged
            return {"message": data["choices"][0]["message"]}

        else:  # ollama (default)
            body = {
                "model":    MODEL,
                "messages": messages,
                "stream":   False,
                "think":    not FLASH_MODE,
            }
            if tools:   body["tools"]  = tools
            if format:  body["format"] = format
            resp = await client.post(f"{OLLAMA_URL}/api/chat", json=body)
            resp.raise_for_status()
            return resp.json()


async def check_llm_backend() -> None:
    """Verify the configured LLM backend is reachable and the model is available."""
    async with httpx.AsyncClient(timeout=10) as client:
        if BACKEND == "lmstudio":
            try:
                r = await client.get(f"{LMSTUDIO_URL}/v1/models")
                r.raise_for_status()
                available = [m["id"] for m in r.json().get("data", [])]
                if any(MODEL in m for m in available):
                    print(f"✓ LM Studio OK — '{MODEL}' found")
                else:
                    print(f"⚠  LM Studio reachable but '{MODEL}' not listed.")
                    print(f"   Available: {available}")
            except Exception as exc:
                print(f"✗ LM Studio unreachable at {LMSTUDIO_URL}: {exc}")
        else:
            try:
                r = await client.get(f"{OLLAMA_URL}/api/tags")
                r.raise_for_status()
                available = [m["name"] for m in r.json().get("models", [])]
                if any(MODEL in m for m in available):
                    print(f"✓ Ollama OK — '{MODEL}' found")
                else:
                    print(f"⚠  Ollama reachable but '{MODEL}' not found locally.")
                    print(f"   Run:  ollama pull {MODEL}")
                    print(f"   Available: {available}")
            except Exception as exc:
                print(f"✗ Ollama unreachable at {OLLAMA_URL}: {exc}")


In [85]:
# ── Tool helpers ──────────────────────────────────────────────────────────────

# Tools that produce screenshots — visibility controlled by use_vision
# Windows-MCP screenshot tool name — discovered at runtime via session.list_tools().
# Common names: "take_screenshot", "screenshot", "windows_screenshot".
# Update this set once you've confirmed the exact tool name.
_SCREENSHOT_TOOLS: set[str] = set()  # populated below after tool discovery

def _update_screenshot_tools(tools: list) -> None:
    """Auto-detect screenshot tools from the Windows-MCP tool list."""
    global _SCREENSHOT_TOOLS
    _SCREENSHOT_TOOLS = {
        t.name for t in tools
        if "screenshot" in t.name.lower() or "snapshot" in t.name.lower()
    }
    if _SCREENSHOT_TOOLS:
        print(f"  [vision] screenshot tools detected: {_SCREENSHOT_TOOLS}")


def filter_tools(tools: list, vision_mode: str) -> list:
    """
    Control which MCP tools are exposed to the model.

    "false"  → remove screenshot tools entirely (model never sees them)
    "true"   → remove screenshot tools (loop injects screenshots automatically;
                exposing the tool would cause double-captures and loop confusion)
    "auto"   → keep all tools (model calls browser_take_screenshot on its own)
    """
    if vision_mode in ("false", "true"):
        return [t for t in tools if t.name not in _SCREENSHOT_TOOLS]
    return list(tools)


def to_ollama_tools(mcp_tools: list) -> list[dict]:
    """Convert MCP Tool objects to Ollama/OpenAI function-calling schema."""
    return [
        {
            "type": "function",
            "function": {
                "name":        t.name,
                "description": t.description or "",
                # inputSchema is already a JSON Schema dict — pass through directly
                "parameters":  t.inputSchema,
            },
        }
        for t in mcp_tools
    ]

In [86]:
# ── Observation & screenshot helpers ─────────────────────────────────────────

# Global frame buffer — populated during a run, stitched into GIF at the end.
# Cleared by main() at the start of each run.
# Sub-agents pass their own list via run_agent(frame_buffer=...) to avoid interleaving.
_gif_frames: list[str] = []


def extract_observation(result, frame_buffer: list | None = None) -> tuple[str, list[str]]:
    """
    Flatten a CallToolResult into (text, [base64_images]).
    Images are displayed inline and collected for GIF replay.

    frame_buffer — per-agent list to append frames to; falls back to global _gif_frames.
    """
    _buf = frame_buffer if frame_buffer is not None else _gif_frames
    text_parts: list[str] = []
    images: list[str] = []

    for item in result.content:
        if item.type == "text":
            text_parts.append(item.text)
        elif item.type == "image":
            images.append(item.data)
            _buf.append(item.data)                 # collect for GIF
            display(IPImage(data=base64.b64decode(item.data)))
        elif item.type == "resource":
            res = item.resource
            text_parts.append(getattr(res, "text", str(getattr(res, "uri", res))))

    text = "\n".join(text_parts).strip() or "(empty)"
    return text, images


async def capture_screenshot(session: ClientSession, frame_buffer: list | None = None) -> str | None:
    """
    Directly call browser_take_screenshot on the MCP session.
    Returns raw base64 PNG, or None on failure.

    frame_buffer — per-agent list to append frames to; falls back to global _gif_frames.
    """
    _buf = frame_buffer if frame_buffer is not None else _gif_frames
    try:
        result = await session.call_tool("browser_take_screenshot", {})
        for item in result.content:
            if item.type == "image":
                _buf.append(item.data)             # collect for GIF
                return item.data
    except Exception as exc:
        print(f"  [screenshot failed] {exc}")
    return None


In [87]:
# ── Run artifacts & GIF replay ────────────────────────────────────────────────

def make_run_dir(prefix: str = "") -> Path:
    """Create a timestamped directory under ./runs/ for this run's outputs.

    prefix — optional string prepended to the timestamp (e.g. "agent0_").
    """
    run_dir = Path("runs") / (prefix + datetime.now().strftime("%Y%m%d_%H%M%S"))
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


async def save_artifacts(
    final_answer: str,
    run_dir: Path,
    steps_taken: int = 0,
    task: str | None = None,
) -> None:
    """
    Ask the LLM to format the result as structured JSON, then save:
      result.json   — machine-readable structured summary
      summary.txt   — plain-text answer

    Args:
      task — the resolved task string for this run (falls back to global TASK
             if not provided, preserving backward compatibility).
    """
    _task = task if task is not None else TASK

    schema = {
        "type": "object",
        "properties": {
            "task":        {"type": "string"},
            "status":      {"type": "string", "enum": ["completed", "failed", "partial"]},
            "summary":     {"type": "string"},
            "steps_taken": {"type": "integer"},
        },
        "required": ["task", "status", "summary"],
    }
    try:
        resp = await llm_chat(
            messages=[
                {"role": "system", "content": "Output ONLY valid JSON matching the schema. No prose."},
                {"role": "user",   "content": f"Task: {_task}\nResult: {final_answer}\nSteps taken: {steps_taken}"},
            ],
            format=schema,
        )
        structured = json.loads(resp["message"].get("content", "{}"))
    except Exception:
        structured = {"task": _task, "status": "completed", "summary": final_answer, "steps_taken": steps_taken}

    (run_dir / "result.json").write_text(json.dumps(structured, indent=2))
    (run_dir / "summary.txt").write_text(final_answer)
    print(f"  [artifacts] Saved → {run_dir}/")
    display(Markdown(f"**Artifacts:** `{run_dir}/result.json` · `{run_dir}/summary.txt`"))


def save_gif(run_dir: Path, frames: list[str] | None = None) -> None:
    """
    Stitch frames into an animated GIF and display it inline.

    frames — explicit frame list (used by sub-agents); falls back to global _gif_frames.
    Skips silently if no frames were collected during the run.
    """
    _frames = frames if frames is not None else _gif_frames
    if not _frames:
        print("  [gif] No frames — skipping")
        return
    try:
        pil_frames = [
            PILImage.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")
            for b64 in _frames
        ]
        gif_path = run_dir / "replay.gif"
        pil_frames[0].save(
            gif_path,
            save_all=True,
            append_images=pil_frames[1:],
            duration=GIF_FRAME_MS,
            loop=0,
            optimize=True,
        )
        print(f"  [gif] {len(pil_frames)} frames → {gif_path}")
        display(IPImage(filename=str(gif_path)))
    except Exception as exc:
        print(f"  [gif] Failed: {exc}")


In [88]:
# ── Human-in-the-loop checkpoints ────────────────────────────────────────────

async def confirm_action(tool_name: str, tool_args: dict) -> bool | None:
    """
    If human_in_loop is enabled and the action matches a confirm keyword,
    pause and show a widget asking the user to approve, skip, or stop the run.

    Returns:
      True  — proceed with the action
      False — skip this action (agent sees a 'skipped' observation)
      None  — stop the entire run
    """
    if not HUMAN_IN_LOOP:
        return True

    action_str = (tool_name + " " + json.dumps(tool_args)).lower()
    if not any(kw.lower() in action_str for kw in CONFIRM_KEYWORDS):
        return True

    confirmed = asyncio.Event()
    result: dict = {"value": True}

    display(Markdown(
        f"⚠️ **Human checkpoint** — action requires approval\n\n"
        f"**Tool:** `{tool_name}`\n"
        f"```json\n{json.dumps(tool_args, indent=2)}\n```"
    ))

    btn_yes  = widgets.Button(description="✓ Proceed",   button_style="success", icon="check")
    btn_skip = widgets.Button(description="↷ Skip",      button_style="warning", icon="forward")
    btn_stop = widgets.Button(description="⏹ Stop run",  button_style="danger",  icon="stop")

    def _yes(b):
        result["value"] = True
        confirmed.set()

    def _skip(b):
        result["value"] = False
        confirmed.set()

    def _stop(b):
        result["value"] = None
        confirmed.set()

    btn_yes.on_click(_yes)
    btn_skip.on_click(_skip)
    btn_stop.on_click(_stop)

    display(widgets.HBox([btn_yes, btn_skip, btn_stop]))
    await confirmed.wait()
    return result["value"]

In [89]:
# ── Session persistence (not used for Windows-MCP) ──────────────────────────
# Windows-MCP doesn't persist browser cookies or localStorage.
# These stubs keep main() compatible without errors.

async def load_session(session) -> None:  # noqa: unused
    pass

async def save_session(session) -> None:  # noqa: unused
    pass


In [90]:
# ── Context compression & retry ──────────────────────────────────────────────

async def maybe_compress(messages: list[dict]) -> list[dict]:
    """
    When the non-system message count exceeds CONTEXT_COMPRESS_AFTER,
    summarise the older half into a single 'memory' message.
    The most recent 10 turns are always kept verbatim.
    """
    if CONTEXT_COMPRESS_AFTER <= 0:
        return messages

    non_system = [m for m in messages if m.get("role") != "system"]
    if len(non_system) < CONTEXT_COMPRESS_AFTER:
        return messages

    system_msgs = [m for m in messages if m.get("role") == "system"]
    keep        = 10
    older       = non_system[:-keep]
    recent      = non_system[-keep:]

    try:
        resp = await llm_chat([
            {"role": "system", "content": "Summarise this conversation concisely. Preserve key facts, observations, URLs, and any data found."},
            {"role": "user",   "content": "\n".join(
                f"[{m['role']}]: {str(m.get('content', ''))[:400]}" for m in older
            )},
        ])
        summary = resp["message"].get("content", "")
    except Exception as exc:
        summary = f"(compression failed: {exc})"

    summary_msg = {"role": "user", "content": f"[History summary — {len(older)} turns]\n{summary}"}
    print(f"  [context] Compressed {len(older)} old turns into summary")
    return system_msgs + [summary_msg] + recent


async def call_tool_safely(session: ClientSession, fn_name: str, fn_args: dict):
    """
    Call an MCP tool with optional retry on failure.
    When retry_enabled is False, failures surface immediately (original behaviour).
    """
    if not RETRY_ENABLED:
        return await session.call_tool(fn_name, fn_args)

    last_exc: Exception | None = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return await session.call_tool(fn_name, fn_args)
        except Exception as exc:
            last_exc = exc
            if attempt < MAX_RETRIES:
                print(f"  [retry {attempt}/{MAX_RETRIES}] {fn_name} failed: {exc}")
                await asyncio.sleep(RETRY_DELAY_S)
    raise last_exc  # type: ignore[misc]

In [91]:
# ── ReAct agent loop ──────────────────────────────────────────────────────────

async def run_agent(
    task: str,
    session: ClientSession,
    system_prompt: str | None = None,
    frame_buffer: list | None = None,   # per-agent GIF frame list; None = use global
) -> tuple[str, int]:
    """
    ReAct loop:  Thought (LLM) → Action (tool call) → Observation → repeat.

    Returns (final_answer, unique_steps_taken).

    Args:
      system_prompt — override the global SYSTEM_PROMPT (used by skills that
                      append a system_prompt_extension); defaults to the global.
      frame_buffer  — per-agent list for screenshot frames; pass a local list
                      when running sub-agents in parallel to prevent frame
                      interleaving. Defaults to the global _gif_frames.

    Features active in this loop (all config-gated):
      • context compression   — trims old history to stay within token limits
      • human-in-the-loop     — pause widget before dangerous actions
      • retry & error recovery — retries failed tool calls with backoff
      • loop detection        — detects repeated action fingerprints; skips
                                re-execution and nudges the model to try differently;
                                only unique actions count against the step budget
      • model timeout         — kills the run if Ollama doesn't respond within
                                MODEL_TIMEOUT_S seconds (0 = disabled)
      • vision auto-capture   — screenshot injected after each action round
      • GIF frame collection  — every screenshot lands in frame_buffer (or _gif_frames)
    """

    # ── Tool discovery ────────────────────────────────────────────────────────
    tools_resp    = await session.list_tools()
    _update_screenshot_tools(tools_resp.tools)
    visible_tools = filter_tools(tools_resp.tools, USE_VISION)
    ollama_tools  = to_ollama_tools(visible_tools)

    print(f"[init] {len(visible_tools)} tools exposed to model:")
    for t in visible_tools:
        print(f"       • {t.name}")

    # ── Seed history ──────────────────────────────────────────────────────────
    _sys = system_prompt if system_prompt is not None else SYSTEM_PROMPT
    messages: list[dict] = [
        {"role": "system", "content": _sys},
        {"role": "user",   "content": task},
    ]

    final_answer  = ""
    unique_steps  = 0                    # steps with novel (non-repeated) actions
    _fp_history:  list[frozenset] = []   # action fingerprints of past steps
    _repeat_streak: int = 0              # consecutive repeat count

    def _action_fingerprint(tcs: list) -> frozenset:
        """Canonical fingerprint for a list of tool calls: frozenset of (name, sorted_args_json)."""
        parts = []
        for tc in tcs:
            fn   = tc["function"]["name"]
            args = tc["function"].get("arguments", {})
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except Exception:
                    args = {}
            parts.append((fn, json.dumps(args, sort_keys=True)))
        return frozenset(parts)

    async def _timed_chat(*args, **kwargs) -> dict:
        """Wrapper around ollama_chat that enforces MODEL_TIMEOUT_S."""
        if MODEL_TIMEOUT_S > 0:
            try:
                return await asyncio.wait_for(
                    llm_chat(*args, **kwargs),
                    timeout=MODEL_TIMEOUT_S,
                )
            except asyncio.TimeoutError:
                raise TimeoutError(
                    f"Model did not respond within {MODEL_TIMEOUT_S}s. "
                    "Increase model_timeout_s in config.json or check Ollama."
                )
        return await llm_chat(*args, **kwargs)

    for step in range(1, MAX_STEPS + 1):
        bar = "═" * 64
        print(f"\n{bar}\n  Step {step} / {MAX_STEPS}  (unique: {unique_steps})\n{bar}")

        # Context compression (if enabled)
        messages = await maybe_compress(messages)

        # ── LLM call ─────────────────────────────────────────────────────────
        try:
            llm_resp = await _timed_chat(messages, tools=ollama_tools)
        except TimeoutError as exc:
            display(Markdown(f"---\n**Run killed — model timeout:** {exc}"))
            return str(exc), unique_steps

        assistant_msg = llm_resp["message"]
        thought       = (assistant_msg.get("content") or "").strip()
        tool_calls    = assistant_msg.get("tool_calls") or []

        if thought:
            display(Markdown(f"**Thought:** {thought}"))

        # ── Terminal condition ────────────────────────────────────────────────
        if not tool_calls:
            final_answer = thought
            display(Markdown(f"---\n**Task complete:**\n\n{thought}"))
            break

        # ── Loop detection ────────────────────────────────────────────────────
        if LOOP_DETECTION:
            fp     = _action_fingerprint(tool_calls)
            window = _fp_history[-LOOP_WINDOW:] if LOOP_WINDOW > 0 else []

            if fp in window:
                _repeat_streak += 1
                display(Markdown(
                    f"⚠ **Loop detected** (repeat {_repeat_streak}/{MAX_LOOP_REPEATS}) — "
                    f"same action(s) already attempted in a recent step."
                ))

                # Inject assistant msg + synthetic tool responses — no real execution
                messages.append(assistant_msg)
                for tc in tool_calls:
                    synthetic: dict = {
                        "role":    "tool",
                        "content": (
                            f"[loop] '{tc['function']['name']}' was already called with "
                            f"identical arguments in a recent step. Re-execution skipped. "
                            f"You must try a different approach."
                        ),
                    }
                    if tc.get("id"):
                        synthetic["tool_call_id"] = tc["id"]
                    messages.append(synthetic)

                if _repeat_streak >= MAX_LOOP_REPEATS:
                    # Hard stop — ask for a summary and return
                    messages.append({"role": "user", "content": (
                        "You have repeated the same actions too many times and made no progress. "
                        "Summarise what you have accomplished so far and stop."
                    )})
                    try:
                        r = await _timed_chat(messages)
                    except TimeoutError as exc:
                        display(Markdown(f"---\n**Run killed — model timeout:** {exc}"))
                        return str(exc), unique_steps
                    final_answer = (r["message"].get("content") or "").strip()
                    display(Markdown(f"---\n**Stopped (loop limit reached):**\n\n{final_answer}"))
                    return final_answer, unique_steps
                else:
                    messages.append({"role": "user", "content": (
                        "You are repeating actions you have already taken. "
                        "The browser state has not changed. "
                        "Try a **different** approach to make progress."
                    )})
                continue   # skip tool execution; go to next iteration

            else:
                _repeat_streak = 0
                _fp_history.append(fp)
        # ─────────────────────────────────────────────────────────────────────

        unique_steps += 1
        messages.append(assistant_msg)

        # ── Execute tool calls ────────────────────────────────────────────────
        for tc in tool_calls:
            fn_name = tc["function"]["name"]
            fn_args = tc["function"].get("arguments", {})
            if isinstance(fn_args, str):
                fn_args = json.loads(fn_args)

            print(f"\n  ▶  {fn_name}")
            if fn_args:
                print(textwrap.indent(json.dumps(fn_args, indent=2), "     "))

            # Human-in-the-loop checkpoint
            approved = await confirm_action(fn_name, fn_args)
            if approved is None:
                final_answer = "Run stopped by user at human checkpoint."
                display(Markdown(f"---\n**Stopped:** {final_answer}"))
                return final_answer, unique_steps
            if not approved:
                print(f"  [human] Skipped {fn_name}")
                tool_msg: dict = {"role": "tool", "content": f"Action '{fn_name}' was skipped by the user."}
                if tc.get("id"):
                    tool_msg["tool_call_id"] = tc["id"]
                messages.append(tool_msg)
                continue

            # Execute (with retry)
            try:
                tool_result          = await call_tool_safely(session, fn_name, fn_args)
                obs_text, obs_images = extract_observation(tool_result, frame_buffer=frame_buffer)
            except Exception as exc:
                obs_text, obs_images = f"ERROR: {exc}", []
                print(f"  [error] {obs_text}")

            preview = obs_text[:400] + ("…" if len(obs_text) > 400 else "")
            print(f"\n  ◀  {preview}")

            tool_msg = {"role": "tool", "content": obs_text}
            if tc.get("id"):
                tool_msg["tool_call_id"] = tc["id"]
            messages.append(tool_msg)

            # If tool returned images (e.g. model-requested screenshot in vision=auto),
            # attach as a user message — Ollama tool messages don't carry images
            if obs_images and USE_VISION != "false":
                messages.append({
                    "role":    "user",
                    "content": f"[Visual output from {fn_name}]",
                    "images":  [obs_images[0]],
                })

        # ── vision="true": auto-capture after every action round ──────────────
        if USE_VISION == "true":
            b64 = await capture_screenshot(session, frame_buffer=frame_buffer)
            if b64:
                print("  [vision] auto-screenshot captured")
                display(IPImage(data=base64.b64decode(b64)))
                messages.append({
                    "role":    "user",
                    "content": "Current browser state after your actions:",
                    "images":  [b64],
                })

    else:
        # MAX_STEPS exhausted
        print(f"\n⚠  Max steps ({MAX_STEPS}) reached.")
        messages.append({"role": "user", "content": "Max steps reached. Summarise what you accomplished so far."})
        try:
            r = await _timed_chat(messages)
        except TimeoutError as exc:
            display(Markdown(f"---\n**Run killed — model timeout:** {exc}"))
            return str(exc), unique_steps
        final_answer = (r["message"].get("content") or "").strip()
        display(Markdown(f"---\n**Summary (max steps):**\n\n{final_answer}"))

    return final_answer, unique_steps


In [92]:
# ── Orchestrator: sub-agent runner + orchestration loop ──────────────────────


async def _run_one_agent(
    index: int,
    task: str,
    context: str = "",
) -> dict:
    """
    Spin up an isolated sub-agent with its own browser (MCP session + Chromium).
    Returns {"index": int, "task": str, "answer": str, "steps": int}.

    context — cumulative results from prior sequential sub-agents; prepended to
              the task so this agent has the full picture.
    """
    full_task = (f"Context from prior steps:\n{context}\n\n{task}") if context else task
    frames: list[str] = []
    run_dir = make_run_dir(prefix=f"agent{index}_") if (SAVE_ARTIFACTS or SAVE_GIF) else None

    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            # Session persistence intentionally skipped for sub-agents to avoid
            # auth-state conflicts when multiple agents run concurrently.
            final_answer, steps = await run_agent(
                full_task, session,
                frame_buffer=frames,
            )

    if run_dir:
        if SAVE_ARTIFACTS:
            await save_artifacts(final_answer, run_dir, steps_taken=steps, task=full_task)
        if SAVE_GIF and frames:
            save_gif(run_dir, frames=frames)

    return {"index": index, "task": task, "answer": final_answer, "steps": steps}


async def run_orchestrator(goal: str) -> str:
    """
    Orchestrator ReAct loop.

    The orchestrator LLM receives the high-level `goal` and a single synthetic
    tool: spawn_subagents.  It may call that tool one or more times, each time
    specifying a list of browser sub-tasks and a mode ("parallel" or "sequential").

    parallel   — all sub-tasks in the batch start at once via asyncio.gather()
    sequential — sub-tasks run in order; each agent receives the accumulated
                 results of all previous agents as context

    The orchestrator synthesises the results and returns a final answer.
    """
    await check_llm_backend()

    spawn_tool = {
        "type": "function",
        "function": {
            "name": "spawn_subagents",
            "description": (
                "Delegate one or more browser tasks to isolated sub-agents. "
                "Each sub-agent gets its own browser instance. "
                "Use mode='parallel' when tasks are independent (faster); "
                "use mode='sequential' when each task should see the previous result."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tasks": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of specific, self-contained browser tasks to delegate.",
                    },
                    "mode": {
                        "type": "string",
                        "enum": ["parallel", "sequential"],
                        "description": (
                            "parallel = all tasks start simultaneously; "
                            "sequential = tasks run in order, each receives prior results as context."
                        ),
                    },
                },
                "required": ["tasks", "mode"],
            },
        },
    }

    messages: list[dict] = [
        {"role": "system", "content": ORCHESTRATOR_SYSTEM_PROMPT},
        {"role": "user",   "content": goal},
    ]

    async def _timed_orch(*args, **kwargs) -> dict:
        if MODEL_TIMEOUT_S > 0:
            try:
                return await asyncio.wait_for(llm_chat(*args, **kwargs), timeout=MODEL_TIMEOUT_S)
            except asyncio.TimeoutError:
                raise TimeoutError(f"Orchestrator model timed out after {MODEL_TIMEOUT_S}s.")
        return await llm_chat(*args, **kwargs)

    for step in range(1, MAX_STEPS + 1):
        bar = "═" * 64
        print(f"\n{bar}\n  Orchestrator step {step} / {MAX_STEPS}\n{bar}")

        try:
            llm_resp = await _timed_orch(messages, tools=[spawn_tool])
        except TimeoutError as exc:
            display(Markdown(f"---\n**Orchestrator killed — timeout:** {exc}"))
            return str(exc)

        assistant_msg = llm_resp["message"]
        thought       = (assistant_msg.get("content") or "").strip()
        tool_calls    = assistant_msg.get("tool_calls") or []

        if thought:
            display(Markdown(f"**Orchestrator thought:** {thought}"))

        # Terminal — no tool calls means the orchestrator is done
        if not tool_calls:
            display(Markdown(f"---\n**Orchestrator final answer:**\n\n{thought}"))
            return thought

        messages.append(assistant_msg)

        for tc in tool_calls:
            fn_args = tc["function"].get("arguments", {})
            if isinstance(fn_args, str):
                fn_args = json.loads(fn_args)

            tasks = fn_args.get("tasks", [])[:MAX_SUBAGENTS]
            mode  = fn_args.get("mode", SUBAGENT_MODE)

            print(f"\n  [orchestrator] Spawning {len(tasks)} sub-agent(s) — mode: {mode}")
            for i, t in enumerate(tasks):
                print(f"    {i}: {t[:120]}")

            if mode == "parallel":
                raw = await asyncio.gather(
                    *[_run_one_agent(i, t) for i, t in enumerate(tasks)],
                    return_exceptions=True,
                )
                results = []
                for r in raw:
                    if isinstance(r, Exception):
                        results.append({"index": len(results), "task": "?", "answer": f"ERROR: {r}", "steps": 0})
                    else:
                        results.append(r)

            else:  # sequential
                results = []
                cumulative = ""
                for i, t in enumerate(tasks):
                    try:
                        r = await _run_one_agent(i, t, context=cumulative)
                    except Exception as exc:
                        r = {"index": i, "task": t, "answer": f"ERROR: {exc}", "steps": 0}
                    results.append(r)
                    cumulative += f"\nSub-agent {i} result:\n{r['answer']}\n"

            obs = "\n\n---\n\n".join(
                f"**Sub-agent {r['index']}** ({r['steps']} steps)\n"
                f"Task: {r['task']}\n"
                f"Result: {r['answer']}"
                for r in results
            )

            tool_msg = {"role": "tool", "content": obs}
            if tc.get("id"):
                tool_msg["tool_call_id"] = tc["id"]
            messages.append(tool_msg)

    # MAX_STEPS exhausted
    messages.append({"role": "user", "content": "Max steps reached. Summarise what you accomplished."})
    try:
        r = await _timed_orch(messages)
    except TimeoutError as exc:
        return str(exc)
    return (r["message"].get("content") or "").strip()


In [93]:
# ── Orchestrator: sub-agent runner + orchestrator loop ───────────────────────

async def _run_one_agent(
    index: int,
    task: str,
    context: str = "",
) -> dict:
    """
    Spin up a fully isolated sub-agent with its own browser (MCP session),
    run directory, and GIF frame buffer.

    Args:
      index   — sub-agent index (used for run dir prefix and logging)
      task    — the specific browser task to perform
      context — optional text from prior sequential sub-agents, prepended to the task

    Returns a dict with keys: index, task, answer, steps
    """
    full_task = (f"Context from prior steps:\n{context}\n\n{task}") if context else task
    frames: list[str] = []
    run_dir = make_run_dir(prefix=f"agent{index}_") if (SAVE_ARTIFACTS or SAVE_GIF) else None

    print(f"  [agent-{index}] Starting: {task[:100]}")
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            final_answer, steps = await run_agent(
                full_task, session,
                frame_buffer=frames,
            )

    print(f"  [agent-{index}] Done ({steps} steps)")

    if run_dir:
        if SAVE_ARTIFACTS:
            await save_artifacts(final_answer, run_dir, steps_taken=steps, task=full_task)
        if SAVE_GIF and frames:
            save_gif(run_dir, frames=frames)

    return {"index": index, "task": task, "answer": final_answer, "steps": steps}


async def run_orchestrator(goal: str) -> str:
    """
    Orchestrator ReAct loop.

    The orchestrator LLM has no direct browser access — it can only call
    spawn_subagents to delegate tasks to isolated sub-agents.  It may call
    spawn_subagents multiple times, mixing parallel and sequential batches.

    Args:
      goal — high-level natural-language goal to decompose and execute

    Returns the orchestrator's final synthesised answer.
    """
    await check_llm_backend()

    spawn_tool = {
        "type": "function",
        "function": {
            "name": "spawn_subagents",
            "description": (
                "Delegate one or more browser tasks to isolated sub-agents. "
                "Each sub-agent gets its own browser instance. "
                "Use mode='parallel' when tasks are independent (faster); "
                "use mode='sequential' when each task should receive the previous result as context."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tasks": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of specific, self-contained browser tasks to delegate.",
                    },
                    "mode": {
                        "type": "string",
                        "enum": ["parallel", "sequential"],
                        "description": (
                            "parallel   = all tasks start simultaneously (best for independent tasks); "
                            "sequential = tasks run in order, each receives prior results as context."
                        ),
                    },
                },
                "required": ["tasks", "mode"],
            },
        },
    }

    messages: list[dict] = [
        {"role": "system", "content": ORCHESTRATOR_SYSTEM_PROMPT},
        {"role": "user",   "content": goal},
    ]

    async def _timed_chat_orch(*args, **kwargs) -> dict:
        if MODEL_TIMEOUT_S > 0:
            try:
                return await asyncio.wait_for(llm_chat(*args, **kwargs), timeout=MODEL_TIMEOUT_S)
            except asyncio.TimeoutError:
                raise TimeoutError(f"Orchestrator model timed out after {MODEL_TIMEOUT_S}s.")
        return await llm_chat(*args, **kwargs)

    for step in range(1, MAX_STEPS + 1):
        bar = "═" * 64
        print(f"\n{bar}\n  Orchestrator step {step} / {MAX_STEPS}\n{bar}")

        try:
            llm_resp = await _timed_chat_orch(messages, tools=[spawn_tool])
        except TimeoutError as exc:
            display(Markdown(f"---\n**Orchestrator killed — timeout:** {exc}"))
            return str(exc)

        assistant_msg = llm_resp["message"]
        thought       = (assistant_msg.get("content") or "").strip()
        tool_calls    = assistant_msg.get("tool_calls") or []

        if thought:
            display(Markdown(f"**Orchestrator thought:** {thought}"))

        # No tool calls = orchestrator is done
        if not tool_calls:
            display(Markdown(f"---\n**Orchestrator final answer:**\n\n{thought}"))
            return thought

        messages.append(assistant_msg)

        for tc in tool_calls:
            fn_args = tc["function"].get("arguments", {})
            if isinstance(fn_args, str):
                fn_args = json.loads(fn_args)

            tasks = fn_args.get("tasks", [])[:MAX_SUBAGENTS]
            mode  = fn_args.get("mode", SUBAGENT_MODE)

            print(f"\n  [orchestrator] Spawning {len(tasks)} sub-agent(s) — mode: {mode}")
            for i, t in enumerate(tasks):
                print(f"    {i}: {t[:120]}")

            if mode == "parallel":
                raw_results = await asyncio.gather(
                    *[_run_one_agent(i, t) for i, t in enumerate(tasks)],
                    return_exceptions=True,
                )
                results: list[dict] = []
                for r in raw_results:
                    if isinstance(r, Exception):
                        results.append({"index": len(results), "task": "?", "answer": f"ERROR: {r}", "steps": 0})
                    else:
                        results.append(r)  # type: ignore[arg-type]

            else:  # sequential
                results = []
                cumulative_context = ""
                for i, t in enumerate(tasks):
                    try:
                        r = await _run_one_agent(i, t, context=cumulative_context)
                    except Exception as exc:
                        r = {"index": i, "task": t, "answer": f"ERROR: {exc}", "steps": 0}
                    results.append(r)
                    cumulative_context += f"\nSub-agent {i} result:\n{r['answer']}\n"

            obs = "\n\n---\n\n".join(
                f"**Sub-agent {r['index']}** ({r['steps']} steps)\n"
                f"Task: {r['task']}\n"
                f"Result: {r['answer']}"
                for r in results
            )
            tool_msg: dict = {"role": "tool", "content": obs}
            if tc.get("id"):
                tool_msg["tool_call_id"] = tc["id"]
            messages.append(tool_msg)

    # MAX_STEPS exhausted — ask for summary
    messages.append({"role": "user", "content": "Max steps reached. Summarise what you accomplished."})
    try:
        r = await _timed_chat_orch(messages)
    except TimeoutError as exc:
        return str(exc)
    return (r["message"].get("content") or "").strip()


In [94]:
# ── Scheduler ─────────────────────────────────────────────────────────────────
# Set "schedule": "<cron expression>" in config.json to run on a repeating schedule.
# Each run is isolated: fresh browser session, new run folder, GIF frames reset.
# Leave "schedule": null (default) to run once via the cell below.
#
# Examples:
#   "* * * * *"    — every minute
#   "0 * * * *"    — every hour
#   "0 9 * * 1-5"  — 09:00 Mon–Fri

_scheduler = None


async def _scheduled_run() -> None:
    """Single isolated run for the scheduler. Supports skills via _resolve_task()."""
    _resolved_task, _effective_prompt = _resolve_task()
    _gif_frames.clear()
    run_dir = make_run_dir() if (SAVE_ARTIFACTS or SAVE_GIF) else None

    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            await load_session(session)
            final_answer, steps = await run_agent(_resolved_task, session, system_prompt=_effective_prompt)
            await save_session(session)

    if run_dir:
        if SAVE_ARTIFACTS:
            await save_artifacts(final_answer, run_dir, steps_taken=steps, task=_resolved_task)
        if SAVE_GIF:
            save_gif(run_dir)


def start_scheduler() -> None:
    from apscheduler.schedulers.asyncio import AsyncIOScheduler
    from apscheduler.triggers.cron import CronTrigger

    global _scheduler
    if _scheduler and _scheduler.running:
        _scheduler.shutdown(wait=False)

    _scheduler = AsyncIOScheduler()
    _scheduler.add_job(
        _scheduled_run,
        CronTrigger.from_crontab(SCHEDULE),
        id="jerry_task",
        max_instances=1,   # prevent overlapping runs
        coalesce=True,
    )
    _scheduler.start()
    label = f"skill:{SKILL}" if SKILL else f'"{TASK[:50]}"'
    print(f"[scheduler] {label} scheduled: {SCHEDULE}")
    print("  Stop with:  stop_scheduler()")


def stop_scheduler() -> None:
    global _scheduler
    if _scheduler:
        _scheduler.shutdown(wait=False)
        print("[scheduler] Stopped")

In [95]:
# ── Run Agent ─────────────────────────────────────────────────────────────────
# • ORCHESTRATOR_TASK set → run orchestrator (decomposes goal → sub-agents)
# • TASK / SKILL set       → run single agent (original behaviour)
# • schedule set           → start cron scheduler (runs in background)

async def main() -> str:
    # Resolve skill → (task, system_prompt) before any I/O.
    # Fails fast with a clear error if SKILL is set but args are missing or the file
    # doesn't exist — before Ollama or the browser are even touched.
    _resolved_task, _effective_prompt = _resolve_task()

    await check_llm_backend()

    _gif_frames.clear()                                          # reset frame buffer
    run_dir = make_run_dir() if (SAVE_ARTIFACTS or SAVE_GIF) else None

    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            final_answer, steps = await run_agent(_resolved_task, session, system_prompt=_effective_prompt)

    if run_dir:
        if SAVE_ARTIFACTS:
            await save_artifacts(final_answer, run_dir, steps_taken=steps, task=_resolved_task)
        if SAVE_GIF:
            save_gif(run_dir)

    return final_answer


if SCHEDULE:
    start_scheduler()
    # Scheduler runs async in the background.
    # Call stop_scheduler() in a new cell to stop it.
elif ORCHESTRATOR_TASK:
    orchestrator_result = asyncio.run(run_orchestrator(ORCHESTRATOR_TASK))
else:
    result = asyncio.run(main())


⚠  LM Studio reachable but 'qwen3-coder:30b' not listed.
   Available: ['qwen/qwen3-coder-30b', 'openai/gpt-oss-20b', 'qwen/qwen3-vl-30b', 'text-embedding-nomic-embed-text-v1.5']

════════════════════════════════════════════════════════════════
  Orchestrator step 1 / 20
════════════════════════════════════════════════════════════════


**Orchestrator thought:** I'll help you compare the prices of these two products by finding their official pricing information. Let me tackle each product separately.

First, I'll check the NVIDIA DGX Spark (Founders Edition) price by searching on Google and visiting the official NVIDIA page.


  [orchestrator] Spawning 1 sub-agent(s) — mode: parallel
    0: Search for 'NVIDIA DGX Spark Founders Edition' on Google and find the official NVIDIA product page link, then navigate t
  [agent-0] Starting: Search for 'NVIDIA DGX Spark Founders Edition' on Google and find the official NVIDIA product page l
[init] 21 tools exposed to model:
       • browser_close
       • browser_resize
       • browser_console_messages
       • browser_handle_dialog
       • browser_evaluate
       • browser_file_upload
       • browser_fill_form
       • browser_install
       • browser_press_key
       • browser_type
       • browser_navigate
       • browser_navigate_back
       • browser_network_requests
       • browser_run_code
       • browser_snapshot
       • browser_click
       • browser_drag
       • browser_hover
       • browser_select_option
       • browser_tabs
       • browser_wait_for

════════════════════════════════════════════════════════════════
  Step 1 / 20  (unique: 0)
═════

**Thought:** I'll help you search for the NVIDIA DGX Spark Founders Edition on Google, find the official product page, and then check its price. Let me break this down into steps:

1. First, I'll navigate to Google
2. Search for "NVIDIA DGX Spark Founders Edition"
3. Find the official NVIDIA product page from the search results
4. Navigate to that page
5. Find the price information

Let me start by navigating to Google:


  ▶  browser_navigate
     {
       "url": "https://www.google.com"
     }

  ◀  ### Ran Playwright code
```js
await page.goto('https://www.google.com');
```
### Page
- Page URL: https://www.google.com/
- Page Title: Google
### Snapshot
```yaml
- generic [ref=e2]:
  - navigation [ref=e3]:
    - link "About" [ref=e4] [cursor=pointer]:
      - /url: https://about.google/?fg=1&utm_source=google-US&utm_medium=referral&utm_campaign=hp-header
    - link "Store" [ref=e5] [cursor=poin…

════════════════════════════════════════════════════════════════
  Step 2 / 20  (unique: 1)
════════════════════════════════════════════════════════════════


...

**Thought:** I'm on the Thinkmate page for NVIDIA DGX Spark, but I need to find the official NVIDIA product page to get the correct pricing. Let me go back to the search results and find a more direct link to NVIDIA's official page.


  ▶  browser_navigate_back

  ◀  ### Ran Playwright code
```js
await page.goBack();
```
### Page
- Page URL: https://www.google.com/search?q=NVIDIA+DGX+Spark+Founders+Edition&sca_esv=129340b3480a109d&source=hp&ei=09a6aabEIaqmqtsPt6OGGA&iflsig=AFdpzrgAAAAAabrk44ko_ltaPQdrd_WlWokIdCdYvNsV&ved=0ahUKEwjm-fWD86mTAxUqk2oFHbeRAQMQ4dUDCBY&uact=5&oq=NVIDIA+DGX+Spark+Founders+Edition&gs_lp=Egdnd3Mtd2l6IiFOVklESUEgREdYIFNwYXJrIEZvdW5kZXJzIE…

════════════════════════════════════════════════════════════════
  Step 9 / 20  (unique: 6)
════════════════════════════════════════════════════════════════


**Thought:** I need to find the official NVIDIA product page for "NVIDIA DGX Spark Founders Edition" and then locate the price on that page.

First, let me search for the product on Google to find the official NVIDIA page.


  ▶  browser_navigate
     {
       "url": "https://www.google.com"
     }

  ◀  ### Ran Playwright code
```js
await page.goto('https://www.google.com');
```
### Page
- Page URL: https://www.google.com/
- Page Title: Google
### Snapshot
```yaml
- generic [ref=e2]:
  - navigation [ref=e3]:
    - link "About" [ref=e4] [cursor=pointer]:
      - /url: https://about.google/?fg=1&utm_source=google-US&utm_medium=referral&utm_campaign=hp-header
    - link "Store" [ref=e5] [cursor=poin…

════════════════════════════════════════════════════════════════
  Step 10 / 20  (unique: 7)
════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════
  Orchestrator step 2 / 20
════════════════════════════════════════════════════════════════


HTTPStatusError: Client error '400 Bad Request' for url 'http://localhost:1234/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400